# Coffee17 EfficientNetV2 progressive multi-granularity

Notebook ini memakai ulang BE2G/BE2H seed 123 dan hanya melatih E2/E3. Output E2/E3 disimpan langsung ke Google Drive dan aman dilanjutkan setelah runtime reset. Screening wajib memakai validation.

In [ ]:
SEEDS = [123]
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
DRIVE_FOLDER = 'coffee17-progressive-efficientnet'
EVALUATION_SPLIT = 'val'

In [ ]:
# SETUP REPO, DRIVE, DATASET, DAN CHECKPOINT BASELINE
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

from google.colab import drive, userdata
drive.mount('/content/drive')

seeds = list(globals().get('SEEDS', [123]))
hf_repo = str(globals().get('HF_REPO', 'ediprin/coffee-backbone-checkpoints'))
drive_folder = str(globals().get('DRIVE_FOLDER', 'coffee17-progressive-efficientnet'))
evaluation_split = str(globals().get('EVALUATION_SPLIT', 'val'))
assert evaluation_split == 'val', 'Jangan buka test pada screening.'

workspace = Path('/content')
repo = workspace / 'coffee-bean-classification'
repo_url = 'https://github.com/ediprin/coffee-bean-classification.git'

def run(command, cwd=None):
    command = [str(item) for item in command]
    print('\n$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

if (repo / '.git').is_dir():
    run(['git', 'pull', '--ff-only'], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', repo_url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)])
assert (repo / 'src/bilinear_lmmd/engine/train_progressive.py').is_file(), 'Repo belum memuat progressive trainer.'

import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

# Dataset direkonstruksi deterministik; checkpoint tetap berada di Drive.
dataset_base = workspace / 'coffee17-progressive-data'
original = dataset_base / 'original'
clean = dataset_base / 'clean'
archive = dataset_base / 'coffee17.zip'
data_root = clean / 'folds/fold_1'
image_suffixes = {'.jpg', '.jpeg', '.png'}

def image_count(path):
    return sum(1 for item in path.rglob('*') if item.is_file() and item.suffix.lower() in image_suffixes) if path.is_dir() else 0

if image_count(original / 'source') != 979:
    if original.exists(): shutil.rmtree(original)
    if archive.exists() and not zipfile.is_zipfile(archive): archive.unlink()
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17', '--output', original, '--archive', archive, '--seed', '42'], cwd=repo)
audit_path = clean / 'audit.json'
valid_clean = False
if audit_path.is_file() and data_root.is_dir():
    try:
        audit = json.loads(audit_path.read_text())
        valid_clean = audit.get('status') == 'complete' and audit.get('clean_count') == 965
    except Exception:
        pass
if not valid_clean:
    if clean.exists(): shutil.rmtree(clean)
    run([sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds', '--source-root', original / 'source', '--output-root', clean, '--expected-count', '979', '--folds', '5', '--seed', '42', '--validation-ratio', '0.1'], cwd=repo)
assert all((data_root / f'source/{split}').is_dir() for split in ('train', 'val', 'test'))
print('DATASET:', data_root)

# Pulihkan BE2G/BE2H tanpa training ulang.
token = userdata.get('HF_TOKEN')
assert token, 'Tambahkan HF_TOKEN ke Colab Secrets.'
os.environ['HF_TOKEN'] = token
from huggingface_hub import snapshot_download
baseline_root = workspace / 'backbone-results'
patterns = []
for seed in seeds:
    patterns += [f'outputs/BE2G_seed{seed}/*', f'outputs/BE2H_seed{seed}/*']
snapshot_download(repo_id=hf_repo, repo_type='model', token=token, local_dir=baseline_root, allow_patterns=patterns)
for seed in seeds:
    for code in ('BE2G', 'BE2H'):
        checkpoint = baseline_root / f'outputs/{code}_seed{seed}/best.pt'
        assert checkpoint.is_file(), f'Checkpoint baseline belum ada: {checkpoint}'

output_root = Path('/content/drive/MyDrive') / drive_folder
output_root.mkdir(parents=True, exist_ok=True)
print('BASELINE:', baseline_root)
print('OUTPUT  :', output_root)

In [ ]:
# TRAIN / RESUME E2 DAN E3 DENGAN PROGRESS
import json, subprocess, sys, time

command = [
    sys.executable, '-u', '-m',
    'bilinear_lmmd.experiments.run_progressive_multigranularity',
    '--data-root', str(data_root),
    '--baseline-root', str(baseline_root),
    '--output-root', str(output_root),
    '--seeds', *[str(seed) for seed in seeds],
    '--evaluation-split', evaluation_split,
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=repo)
started = time.monotonic()
next_heartbeat = started
while process.poll() is None:
    now = time.monotonic()
    if now >= next_heartbeat:
        rows = []
        for code in ('E2', 'E3'):
            for seed in seeds:
                history = output_root / f'outputs/{code}_seed{seed}/history.json'
                if history.is_file():
                    try: rows.append(f'{code}-s{seed}={len(json.loads(history.read_text()))}/50')
                    except Exception: rows.append(f'{code}-s{seed}=saving')
        detail = ', '.join(rows) if rows else 'inisialisasi/download model'
        print(f'[HEARTBEAT {(now-started)/60:.1f} menit] {detail}', flush=True)
        next_heartbeat = now + 60
    time.sleep(5)
return_code = process.wait()
assert return_code == 0, f'Runner berhenti dengan kode {return_code}; lihat error di atas.'
print('SELESAI:', output_root)

In [ ]:
# TAMPILKAN PUTUSAN TANPA TRAINING ULANG
import json
decision_path = output_root / 'val_reports/progressive_decision.json'
assert decision_path.is_file(), f'Belum ada: {decision_path}'
decision = json.loads(decision_path.read_text())
print('=== PUTUSAN PROGRESSIVE MULTI-GRANULARITY ===')
for name in ('E2_vs_GAP', 'E3_vs_GAP', 'E3_vs_E2'):
    row = decision[name]
    print(f"{name:12s}: {row['decision']} | {row['criteria']}")

## Setelah runtime reset

Jalankan kembali semua cell. Dataset dan baseline dipulihkan, sedangkan E2/E3 melanjutkan `last.pt` dari Google Drive. Jangan mengubah split menjadi test sebelum model dan hyperparameter dikunci.